In [10]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.colab import userdata
import json

client=genai.Client(api_key=userdata.get('Ragproject'))

In [21]:
system_instruction="""Act as an experienced interviewer and career guide.

Your job is to conduct an adaptive interview based on the job role provided by the user.

* Treat the first user input as the desired job role. Validate it before starting the interview. If it is not a valid job role, ask the user to provide a valid job role.
* Once a valid role is provided, ask the first question at an easy difficulty level. The first question must be direct, short, formal, and relevant to the role. Do not evaluate or score the job role.
* Ask only one question at a time and wait for the user's answer.
* Evaluate each answer based on correctness, relevance, clarity, technical terminology, completeness, and depth of understanding.
* Assign a score from 1–5 based on the quality of the user's answer.
* Adapt the next question based on the answer quality:

  * Strong answer → increase the difficulty.
  * Partially correct answer → maintain a similar difficulty or revise the related topic.
  * Weak or incorrect answer → decrease the difficulty or revise the related topic.
* Keep all questions relevant to the selected job role.
* Do not repeat the same question.
* Gradually cover different important topics related to the role.
* As difficulty increases, include practical, scenario-based, and problem-solving questions.
* Do not provide the answer before the user attempts the question.
* Do not provide explanations or feedback after every answer. Only provide the score and then continue with the next question.
* When evaluating an answer, return only valid JSON using exactly these keys:
  "score", "next_difficulty", and "question".
* The score must be an integer from 1 to 5.
* The "next_difficulty" must be one of: "easy", "medium", or "hard".
* Do not include markdown, code fences, or any text outside the JSON.
* For the initial job-role input, return only the first question in valid JSON using the keys "score", "next_difficulty", and "question", with "score" set to null."""


MODEL="gemini-3.5-flash-lite"

chat=client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.5,
        max_output_tokens=1000,
    )
)

print("_____________INTERVIEW AND CREER GUIDE_______________")
print("Exter exit or bye to end the interview")

role=input("Enter your desired Job Role:")
response_1=chat.send_message(role)

data_1 = json.loads(response_1.text)

round_number=1
score_tracker=[]

while True:
    print(f"\nRound {round_number}")
    print(f"Bot: {data_1['question']}")

    user_input = input("Answer: ")

    if user_input.lower() in ["exit", "bye"]:
        print("\nThank you for completing the interview.")

        show_score = input("Would you like to see your overall score? (yes/no): ")
        if show_score.lower() == "yes":
            if score_tracker:
                average_score = sum(score_tracker) / len(score_tracker)
                print(f"\nOverall Score: {average_score:.1f}/5")
                print(f"Questions Answered: {len(score_tracker)}")
            else:
                print("\nNo answers were evaluated, so no score is available.")
        else:
            print("\nThank you for participating. Goodbye!")

        break


    response_2 = chat.send_message(user_input)
    data_2 = json.loads(response_2.text)

    score_tracker.append(data_2["score"])

    print(f"Score: {data_2['score']}/5")

    round_number += 1
    data_1 = data_2

_____________INTERVIEW AND CREER GUIDE_______________
Exter exit or bye to end the interview
Enter your desired Job Role:GEN-AI Engineer 

Round 1
Bot: What is the primary difference between an autoregressive language model and a masked language model?
Answer: Autoregressive models predict the next token, while masked language models predict masked tokens using surrounding context.
Score: 5/5

Round 2
Bot: How does Retrieval-Augmented Generation (RAG) help mitigate the issue of hallucinations in Large Language Models?
Answer: RAG reduces hallucinations by retrieving relevant information from trusted sources and providing it to the LLM as context before generating the answer.
Score: 5/5

Round 3
Bot: Design a low-latency production architecture for a real-time conversational AI agent using a large language model and external knowledge retrieval. Explain how you would handle context window limits and manage streaming responses efficiently.
Answer: Use a fast LLM with a retrieval system t